# Solver environment check

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/aabw/notebooks/lecture-2/jeff-kantor-solver-installation.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/aabw/notebooks/lecture-2/jeff-kantor-solver-installation.ipynb)

> A maintained open-source solver check for the course. Jeff Kantor's original multi-solver installer is retained in the repository [`archive/`](../../../../archive/jeff-kantor-solver-installation-legacy.ipynb).


## Inspect first, install when needed
This companion keeps Jeff Kantor's original installer in the linked historical reference and explains a current approach. We first inspect the environment, then install Pyomo, then HiGHS, and finally Ipopt/CBC. No installation happens merely by importing the shared module.

The function bodies are visible here because they are the subject of the lesson. In notebooks that just use these operations, import them from `support/teaching_utils.py`.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

from importlib import metadata
import importlib.util
import subprocess
from teaching_utils import make_solver, install_coin_solvers, solve_checked


## Installed distributions
`importlib.metadata` reads installed distribution metadata. A distribution name (for example `scikit-learn`) can differ from its import name (`sklearn`). A list of installed packages does not prove that all imports or solvers will run.


In [ ]:
def installed_packages():
    """Return installed distribution names and versions, sorted by name."""
    return dict(sorted(
        ((dist.metadata['Name'], dist.version) for dist in metadata.distributions()
         if dist.metadata['Name']), key=lambda item: item[0].casefold()))

packages_before = installed_packages()
print(f'{len(packages_before)} distributions installed')
print(packages_before)


## Install only a missing dependency
Use the current kernel's Python interpreter for pip. Existing packages remain in place. Here we introduce Pyomo by itself. It expresses a mathematical model but does not supply an optimization solver.


In [ ]:
def ensure_packages(required_packages):
    """Install only missing imports using this kernel's Python; never upgrade."""
    missing = [package for name, package in required_packages.items()
               if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    return missing

ensure_packages({'pyomo': 'pyomo'})
import pyomo.environ as pyo


## Inspect a selection of solver interfaces
Pyomo can test named interfaces directly. This function checks a course selection, not every plugin. `appsi_highs` and `highs` are interfaces to the same engine. Availability checks are followed by actual solves below.


In [ ]:
def available_pyomo_solvers(candidates=None):
    """Check named interfaces, not solver licences or mathematical capabilities.

    The default is a course-oriented selection, not every Pyomo plugin.
    Two interfaces such as highs/appsi_highs are the same solver engine.
    """
    if candidates is None:
        candidates = ('appsi_highs', 'highs', 'ipopt', 'cbc', 'glpk',
                      'gurobi_direct', 'cplex_direct', 'mosek', 'scip', 'xpress')
    result = {}
    for name in candidates:
        try:
            result[name] = bool(make_solver(name).available(exception_flag=False))
        except (ImportError, RuntimeError, ValueError, AttributeError):
            result[name] = False
    return result

print(available_pyomo_solvers())


## Add HiGHS now
Only at this point do we require a solver. Rerunning this cell is harmless when HiGHS is already installed.


In [ ]:
ensure_packages({'highspy': 'highspy'})
print(available_pyomo_solvers(['appsi_highs']))


In [ ]:
def create_model():
    model = pyo.ConcreteModel('solver_check')
    model.x = pyo.Var(domain=pyo.NonNegativeReals)
    model.y = pyo.Var(domain=pyo.NonNegativeReals)
    model.capacity = pyo.Constraint(expr=model.x + model.y <= 15)
    model.objective = pyo.Objective(expr=model.x + 2 * model.y, sense=pyo.maximize)
    return model

model = create_model()
solve_checked(model, 'appsi_highs')
assert abs(pyo.value(model.objective) - 30) < 1e-6
model.display()


## Add two different solver engines
The [AMPL COIN module](https://dev.ampl.com/ampl/python/modules.html) supplies open-source Ipopt and CBC executables. This is a separate installation step. `make_solver` selects the appropriate Pyomo interface for each executable. We solve a fresh model with each engine and check the result; an unavailable solver must not silently count as a comparison.


In [ ]:
install_coin_solvers()
print(available_pyomo_solvers(['ipopt', 'cbc', 'appsi_highs']))


In [ ]:
from time import perf_counter
comparison = []
for name in ('ipopt', 'cbc', 'appsi_highs'):
    model = create_model()
    started = perf_counter()
    result = solve_checked(model, name)
    elapsed = perf_counter() - started
    objective = pyo.value(model.objective)
    assert abs(objective - 30) < 1e-4
    comparison.append({'solver': name, 'objective': objective, 'seconds': elapsed})
assert len({row['solver'] for row in comparison}) == 3
print(comparison)


## Clearing variable values
The following function handles nested model blocks and preserves fixed decisions. It does not clear a persistent solver's internal state. For independent timings above, we create both a new model and a new solver for every run.


In [ ]:
def reset_model(model):
    """Clear unfixed variable values, including variables inside nested blocks.

    This does not clear a persistent solver's state. For independent timings,
    create a fresh model and solver for every run.
    """
    import pyomo.environ as pyo
    for variable in model.component_data_objects(pyo.Var, descend_into=True):
        if not variable.fixed:
            variable.set_value(None)

reset_model(model)
assert model.x.value is None and model.y.value is None


Compare `installed_packages()` with the first inventory. If this runtime already contained a package, its installation step did not add it. These cells remain separate to explain when each dependency becomes necessary. Additional solvers may require their own installations or licences; an interface name alone does not establish that a model can be solved.
